In [ ]:
!pip install -U llama-cloud-services llama-index python-dotenv llama-index-llms-azure-openai llama-index-embeddings-azure-openai llama-index-embeddings-huggingface llama-index-readers-file  # parse the document using LlamaParse

### ℹ️ SDK note
This notebook uses **`llama-cloud-services`** (`from llama_cloud_services import LlamaParse`). The older **`llama-parse`** package is deprecated, and `llama_cloud_services` is a drop-in replacement that keeps the same `LlamaParse` API (`result_type`, `system_prompt`, `output_tables_as_HTML`, `load_data`, `get_json_result`).

> LlamaIndex has since released a newer **`llama-cloud`** SDK (Parse API v2) with a different, structured interface (`client.parsing.parse(...)`, `input_options`/`output_options`/`expand`). It's the long-term recommended path; see the official migration guide. We stay on `llama-cloud-services` here so the workshop flow (LlamaParse + `SimpleDirectoryReader`) is unchanged.

**Before running:** set your real `LLAMA_CLOUD_API_KEY` and Azure values in the configuration cell. The keys that were hardcoded here have been removed — rotate any that were shared.


## Import the required libraries

In [ ]:
import os  # import required libraries

os.environ['LLAMA_CLOUD_API_KEY'] = "llx-pGifrNLzIbgpIhrvnVM"  # set environment variable (API key, etc.)
os.environ['AZURE_OPENAI_API_KEY'] = "FkrrJQQJ99CGACYeBjFXJ3w3AAABACOGKkmt"  # set environment variable (API key, etc.)
os.environ['AZURE_OPENAI_ENDPOINT'] = "https:/.com/"  # set environment variable (API key, etc.)
os.environ['AZURE_OPENAI_API_VERSION'] = "20eview"  # pick the API version your Azure resource supports  # set environment variable (API key, etc.)
os.environ['AZURE_OPENAI_DEPLOYMENT_NAME'] = "gpt-5"  # set environment variable (API key, etc.)



llamaparse_api_key = os.getenv("LLAMA_CLOUD_API_KEY")  # parse the document using LlamaParse
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
azure_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview")



## 📄 Parsing a PDF into MD using LlamaParse

In this example, we use **LlamaParse** to convert a PDF document into **Markdown**. LlamaParse analyzes the document structure and extracts the content while preserving elements such as headings, paragraphs, tables, and lists.

The parsed Markdown can be used for downstream tasks such as Retrieval-Augmented Generation (RAG), semantic search, summarization, and question answering.

### 🔍 What this code does

- Initializes the `LlamaParse` parser.
- Loads the PDF document (`2408.09869v5.pdf`).
- Parses the document into Markdown format.
- Combines the parsed pages into a single Markdown document.
- Saves the extracted content as a `.md` file.

---

### 🔄 Parsing Workflow

```text
PDF Document
      │
      ▼
LlamaParse
      │
      ▼
Markdown Output
      │
      ▼
Saved as .md File
```

---

### 📌 Why use LlamaParse?

LlamaParse is designed to extract structured content from documents while preserving their layout. Compared to traditional PDF text extraction, it provides cleaner and more organized output, making it suitable for LLM-based applications.

The generated Markdown can be directly used for:

- Retrieval-Augmented Generation (RAG)
- Semantic Search
- Question Answering
- Document Summarization
- Knowledge Base Creation

In [ ]:
import os
from llama_cloud_services import LlamaParse
from llama_index.core import SimpleDirectoryReader

# Get the LlamaParse API key from the environment
llamaparse_api_key = os.getenv("LLAMA_CLOUD_API_KEY")

# Initialize the LlamaParse parser
parser = LlamaParse(
    api_key=llamaparse_api_key,
    result_type="markdown",
    verbose=True
)

# Input PDF
input_file = "/content/2408.09869v5.pdf"

# Register the parser for PDF files
file_extractor = {
    ".pdf": parser
}

# Parse the document
documents = SimpleDirectoryReader(
    input_files=[input_file],
    file_extractor=file_extractor
).load_data()

# Create output directory
output_dir = "parsed_output_simple"
os.makedirs(output_dir, exist_ok=True)

# Output markdown file
output_file = os.path.join(
    output_dir,
    f"{os.path.splitext(os.path.basename(input_file))[0]}_parsed.md"
)

# Combine all parsed pages into a single markdown document
markdown_content = "\n\n".join(doc.text for doc in documents)

# Save the parsed markdown
with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Parsed markdown saved to: {output_file}")

/tmp/ipykernel_13331/3102657136.py:2: DeprecationWarning: This package (llama-cloud-services) is deprecated and will be maintained until May 1, 2026. Please migrate to the new package: pip install llama-cloud>=1.0 (https://github.com/run-llama/llama-cloud-py). The new package provides the same functionality with improved performance and support.
  from llama_cloud_services import LlamaParse


Started parsing the file under job_id 0f623264-2e58-454a-8f33-9298251265f1
Parsed markdown saved to: parsed_output_simple/2408.09869v5_parsed.md


## 📄 Parsing a PDF into JSON using LlamaParse

In this example, we use **LlamaParse** to convert a PDF document into a structured **JSON** format. Unlike Markdown, JSON preserves the document in a machine-readable structure, making it easier to process programmatically.

### 🔍 What this code does

- Initializes the `LlamaParse` parser.
- Loads the PDF document (`2408.09869v5.pdf`).
- Parses the document into JSON format.
- Saves the parsed output as a `.json` file.

---

### 🔄 Parsing Workflow

```text
PDF Document
      │
      ▼
LlamaParse
      │
      ▼
Structured JSON
      │
      ▼
Saved as .json File
```

---

### 📌 Why use JSON output?

JSON provides a structured representation of the document, making it useful for applications that require programmatic access to document elements such as:

- Pages
- Headings
- Paragraphs
- Tables
- Metadata
- Layout information

This structured format is commonly used for document processing pipelines, data extraction, and Retrieval-Augmented Generation (RAG) workflows.

In [ ]:
import os
import json
from llama_cloud_services import LlamaParse

# Initialize the LlamaParse parser
parser = LlamaParse(
    api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
    result_type="json",
    verbose=True
)

# Parse the PDF and return the result as JSON
json_result = parser.get_json_result("/content/2408.09869v5.pdf")

# Save the JSON output
output_file = "/content/parsed_output.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(json_result, f, indent=4)

print(f"Parsed JSON saved to: {output_file}")

Started parsing the file under job_id 0f623264-2e58-454a-8f33-9298251265f1
Parsed JSON saved to: /content/parsed_output.json


## 📝 Parsing a PDF with a Custom System Prompt

In this example, we use **LlamaParse** with a **custom system prompt** to control how the parsed document is formatted.

Instead of using the default parsing behavior, we instruct LlamaParse to return the document as **Markdown** while converting all tables into **HTML**. This allows the output to be customized based on the requirements of a specific application.

### 🔍 What this code does

- Initializes `LlamaParse` with a custom system prompt.
- Loads the PDF document (`2408.09869v5.pdf`).
- Parses the document into Markdown format.
- Converts tables into HTML as instructed by the prompt.
- Saves the parsed output as a Markdown file.

---

### 🔄 Parsing Workflow

```text
PDF Document
      │
      ▼
LlamaParse
(Custom System Prompt)
      │
      ▼
Markdown Output
(with HTML Tables)
      │
      ▼
Saved as .md File
```

---

### 📌 Why use a System Prompt?

A system prompt allows you to customize how LlamaParse formats the parsed document without changing the parsing workflow.

Some common customizations include:

- Formatting tables as HTML or Markdown
- Preserving document structure
- Extracting specific sections
- Controlling the formatting of the output

This flexibility makes LlamaParse adaptable to different document-processing and LLM workflows.

In [ ]:
import os

from llama_cloud_services import LlamaParse
from llama_index.core import SimpleDirectoryReader

# Initialize the LlamaParse parser
parser = LlamaParse(
    api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
    result_type="markdown",
    system_prompt="""
      Output the document as Markdown.

      For every figure you encounter:
      - Add a short description.
      - Summarize its purpose in one sentence.

      For every table:
      - Add a one-line summary before the table explaining what it contains.
    """,
    verbose=True
)

# Input PDF
input_file = "/content/2408.09869v5.pdf"

# Register the parser for PDF files
file_extractor = {
    ".pdf": parser
}

# Parse the document
documents = SimpleDirectoryReader(
    input_files=[input_file],
    file_extractor=file_extractor
).load_data()

# Create output directory
output_dir = "parsed_output_simple_json"
os.makedirs(output_dir, exist_ok=True)

# Output markdown file
output_file = os.path.join(
    output_dir,
    f"{os.path.splitext(os.path.basename(input_file))[0]}_parsed.md"
)

# Combine all parsed pages
markdown_content = "\n\n".join(doc.text for doc in documents)

# Save the parsed markdown
with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Parsed markdown saved to: {output_file}")

Started parsing the file under job_id 3887ec21-8c6e-4c41-9a52-c07a2324e724
Parsed markdown saved to: parsed_output_simple_json/2408.09869v5_parsed.md


## 🖼️ Parsing an Image using LlamaParse

In this example, we use **LlamaParse** to extract text and structure from an image document. LlamaParse analyzes the image using OCR and document understanding techniques, then converts the extracted content into **Markdown**.

### 🔍 What this code does

- Initializes the `LlamaParse` parser.
- Loads the image (`file-7mNhEGHpd0.png`).
- Parses the image into Markdown format.
- Saves the extracted content as a Markdown file.

---

### 🔄 Parsing Workflow

```text
Image Document
      │
      ▼
LlamaParse
      │
      ▼
Markdown Output
      │
      ▼
Saved as .md File
```

---

### 📌 Why use LlamaParse for Images?

LlamaParse can process image-based documents such as scanned forms, receipts, invoices, and screenshots. It extracts the visible text while preserving the document's structure, making the output suitable for applications like:

- OCR and text extraction
- Retrieval-Augmented Generation (RAG)
- Semantic Search
- Question Answering
- Document Summarization

---

### 💡 What happens if the image contains both text and pictures?

LlamaParse extracts the visible text from the image while also recognizing the document structure. If the image contains figures, diagrams, or photographs, it may preserve their captions or references, and with a suitable **system prompt**, it can generate brief descriptions or summaries of those visual elements.

In [ ]:
import os
from llama_cloud_services import LlamaParse
from llama_index.core import SimpleDirectoryReader

# Get the LlamaParse API key from the environment
llamaparse_api_key = os.getenv("LLAMA_CLOUD_API_KEY")

# Initialize the LlamaParse parser
parser = LlamaParse(
    api_key=llamaparse_api_key,
    # preserve_layout_alignment_across_pages=True,
    # disable_ocr=True,
    # disable_image_extraction=True,
    # merge_tables_across_pages_in_markdown=True,
    result_type="markdown",
    verbose=True
)

# Input image
input_file = "/content/file-7mNhEGHpd0.png"

# Register the parser for PNG files
file_extractor = {
    ".png": parser
}

# Parse the image
documents = SimpleDirectoryReader(
    input_files=[input_file],
    file_extractor=file_extractor
).load_data()

# Create output directory
output_dir = "parsed_output_png"
os.makedirs(output_dir, exist_ok=True)

# Output markdown file
output_file = os.path.join(
    output_dir,
    f"{os.path.splitext(os.path.basename(input_file))[0]}_parsed.md"
)

# Combine parsed content
markdown_content = "\n\n".join(doc.text for doc in documents)

# Save the parsed markdown
with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Parsed markdown saved to: {output_file}")

Started parsing the file under job_id cc25f49f-a7f7-49e3-9073-c3faf8254e7c
Parsed markdown saved to: parsed_output_png/file-7mNhEGHpd0_parsed.md


## 📊 Parsing a PDF with Tables as HTML

In this example, we configure **LlamaParse** to preserve tables by converting them into **HTML** while generating the rest of the document in **Markdown**. This helps retain the original table structure, making it more suitable for documents that contain complex or large tables.

### 🔍 What this code does

- Initializes `LlamaParse` with HTML table support enabled.
- Loads the PDF document (`2408.09869v5.pdf`).
- Parses the document into Markdown.
- Converts all detected tables into HTML.
- Saves the parsed output as a Markdown file.

---

### 🔄 Parsing Workflow

```text
PDF Document
      │
      ▼
LlamaParse
(HTML Table Extraction)
      │
      ▼
Markdown Output
(with HTML Tables)
      │
      ▼
Saved as .md File
```

---

### 📌 Why output tables as HTML?

While Markdown tables work well for simple data, they have limitations when handling complex table structures such as merged cells, nested tables, or multi-row headers.

Using HTML preserves the table layout more accurately, making it useful for:

- Financial reports
- Research papers
- Insurance forms
- Technical documentation
- Documents with complex tables

In [ ]:
import os
from llama_cloud_services import LlamaParse
from llama_index.core import SimpleDirectoryReader

# Get the LlamaParse API key from the environment
llamaparse_api_key = os.getenv("LLAMA_CLOUD_API_KEY")

# Initialize the LlamaParse parser
parser = LlamaParse(
    api_key=llamaparse_api_key,
    output_tables_as_HTML=True,
    result_type="markdown",
    verbose=True
)

# Input PDF
input_file = "/content/2408.09869v5.pdf"

# Register the parser for PDF files
file_extractor = {
    ".pdf": parser
}

# Parse the document
documents = SimpleDirectoryReader(
    input_files=[input_file],
    file_extractor=file_extractor
).load_data()

# Create output directory
output_dir = "parsed_output_simple_table_html"
os.makedirs(output_dir, exist_ok=True)

# Output markdown file
output_file = os.path.join(
    output_dir,
    f"{os.path.splitext(os.path.basename(input_file))[0]}_parsed.md"
)

# Combine all parsed pages
markdown_content = "\n\n".join(doc.text for doc in documents)

# Save the parsed markdown
with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Parsed markdown saved to: {output_file}")

Started parsing the file under job_id 81ba93ee-1ae0-41ce-ab58-24ee72563059
Parsed markdown saved to: parsed_output_simple_table_html/2408.09869v5_parsed.md


## 🤖 Building a Simple RAG Pipeline with LlamaParse and LlamaIndex

In this example, we combine **LlamaParse** and **LlamaIndex** to build a simple Retrieval-Augmented Generation (RAG) pipeline for a PDF document.

The PDF is first parsed into structured Markdown using LlamaParse. The parsed content is then indexed using LlamaIndex, enabling natural language questions to be answered using the information contained within the document.

### 🔍 What this code does

- Configures the Azure OpenAI LLM.
- Configures a local Hugging Face embedding model.
- Parses the PDF (`2408.09869v5.pdf`) into Markdown using LlamaParse.
- Builds a vector index from the parsed document.
- Creates a query engine.
- Answers questions using the indexed document.

---

### 🔄 RAG Workflow

```text
PDF Document
       │
       ▼
LlamaParse
       │
       ▼
Parsed Markdown
       │
       ▼
Embedding Model
       │
       ▼
VectorStoreIndex
       │
       ▼
Query Engine
       │
       ▼
Azure OpenAI GPT
       │
       ▼
Final Response
```

---

### 📌 Pipeline Components

- **LlamaParse** – Extracts structured content from the PDF and converts it into Markdown.
- **Hugging Face Embedding Model** – Converts the parsed text into vector embeddings.
- **VectorStoreIndex** – Stores the embeddings for efficient semantic retrieval.
- **Query Engine** – Retrieves the most relevant document content based on the user's question.
- **Azure OpenAI GPT** – Generates the final answer using the retrieved context.

---

### 📌 Why build a RAG pipeline?

Instead of asking the LLM to answer without context, a RAG pipeline first retrieves the most relevant information from the indexed document and then uses it to generate a response. This grounds the model's answers in the document, improving accuracy, reducing hallucinations, and enabling users to query large PDF documents using natural language.

## Document Input

In [ ]:
import os
import time
import nest_asyncio

nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
)
from llama_cloud_services import LlamaParse

# ============================================================
# Configure Azure OpenAI
# ============================================================

llm = AzureOpenAI(
    engine=azure_deployment,
    api_key=azure_api_key,
    azure_endpoint=azure_endpoint,
    api_version=azure_api_version,
    temperature=1,
)

# ============================================================
# Configure Embedding Model
# ============================================================

print("Loading embedding model...")

start = time.time()

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(f"Embedding model loaded in {time.time() - start:.2f} seconds")

Settings.llm = llm
Settings.embed_model = embed_model

# ============================================================
# Initialize LlamaParse
# ============================================================

parser = LlamaParse(
    api_key=llamaparse_api_key,
    result_type="markdown",
)

# ============================================================
# Input PDF
# ============================================================

input_file = "/content/2408.09869v5.pdf"

file_extractor = {
    ".pdf": parser
}

# ============================================================
# Parse PDF
# ============================================================

print("\nParsing PDF...")

start = time.time()

documents = SimpleDirectoryReader(
    input_files=[input_file],
    file_extractor=file_extractor,
).load_data()

print(f"PDF parsed in {time.time() - start:.2f} seconds")

# ============================================================
# Save Parsed Markdown
# ============================================================

output_dir = "parsed_output_pdf"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    f"{os.path.splitext(os.path.basename(input_file))[0]}_parsed.md",
)

markdown_content = "\n\n".join(
    doc.text for doc in documents
)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Parsed Markdown saved to: {output_file}")

# ============================================================
# Build Vector Index
# ============================================================

print("\nBuilding vector index...")

start = time.time()

index = VectorStoreIndex.from_documents(documents)

print(f"Vector index built in {time.time() - start:.2f} seconds")

# ============================================================
# Create Query Engine
# ============================================================

query_engine = index.as_query_engine()

# ============================================================
# Sample Queries
# ============================================================

queries = [
    "What is the main topic of this paper?",
    "Who are the authors?",
    "Summarize the document.",
    "What methodology is proposed?",
    "What are the key findings?",
]

# ============================================================
# Execute Queries
# ============================================================

for i, query in enumerate(queries, start=1):

    print("\n" + "=" * 80)
    print(f"Question {i}: {query}")
    print("-" * 80)

    start = time.time()

    response = query_engine.query(query)

    print("Answer:")
    print(response)

    print(f"\nAnswered in {time.time() - start:.2f} seconds")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded in 2.28 seconds

Parsing PDF...
Started parsing the file under job_id 0f623264-2e58-454a-8f33-9298251265f1
PDF parsed in 4.07 seconds
Parsed Markdown saved to: parsed_output_pdf/2408.09869v5_parsed.md

Building vector index...
Vector index built in 2.82 seconds

Question 1: What is the main topic of this paper?
--------------------------------------------------------------------------------
Answer:
An open-source technical report introducing Docling, a self-contained PDF-to-structured-data conversion tool that uses specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), designed to run efficiently on commodity hardware.

Answered in 7.26 seconds

Question 2: Who are the authors?
--------------------------------------------------------------------------------
Answer:
Christoph Auer, Maksym Lysak, Ahmed Nassar, Michele Dolfi, Nikolaos Livathinos, Panos Vagenas, Cesar Berrospi Ramis, Matteo Omenetti, Fabian Lindlbauer, Ka

## Image Input

In [ ]:
import os
import nest_asyncio

nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
)
from llama_cloud_services import LlamaParse

# Configure Azure OpenAI
llm = AzureOpenAI(
    engine=azure_deployment,
    api_key=azure_api_key,
    azure_endpoint=azure_endpoint,
    api_version=azure_api_version,
    temperature=1
)

# Configure embedding model
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Set global LlamaIndex settings
Settings.llm = llm
Settings.embed_model = embed_model

# Initialize LlamaParse
parser = LlamaParse(
    api_key=llamaparse_api_key,
    result_type="markdown"
)

# Parse the image
input_file = "/content/file-7mNhEGHpd0.png"

file_extractor = {
    ".png": parser
}

documents = SimpleDirectoryReader(
    input_files=[input_file],
    file_extractor=file_extractor
).load_data()

# Save parsed markdown
output_dir = "parsed_output_image_11"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    f"{os.path.splitext(os.path.basename(input_file))[0]}_parsed.md"
)

markdown_content = "\n\n".join(doc.text for doc in documents)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Parsed markdown saved to: {output_file}")

# Build a vector index
index = VectorStoreIndex.from_documents(documents)

# Create a query engine
query_engine = index.as_query_engine()

# Sample queries
queries = [
    "Who is the patient?",
    "What is the insurance company name?",
    "What is the patient's address?",
    "What is the patient's date of birth?",
    "What is the policy or group number?"
]

# Execute the queries
for i, query in enumerate(queries, start=1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {query}")
    print("-"*80)

    response = query_engine.query(query)

    print("Answer:")
    print(response)

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Started parsing the file under job_id cc25f49f-a7f7-49e3-9073-c3faf8254e7c
Parsed markdown saved to: parsed_output_image_11/file-7mNhEGHpd0_parsed.md

Question 1: Who is the patient?
--------------------------------------------------------------------------------
Answer:
Mariska Jones

Question 2: What is the insurance company name?
--------------------------------------------------------------------------------
Answer:
United Healthcare

Question 3: What is the patient's address?
--------------------------------------------------------------------------------
Answer:
967 Elm Street, Austin, TX 73301

Question 4: What is the patient's date of birth?
--------------------------------------------------------------------------------
Answer:
01/01/00

Question 5: What is the policy or group number?
--------------------------------------------------------------------------------
Answer:
12345678910
